# 6a · Build an MCP server for colony systems

**Core · Live-code · ~25 min**

An MCP **server** publishes tools and resources over a standard protocol so any client can use
them. Ours already exists in [`orbital_mcp_server.py`](orbital_mcp_server.py). In this beat
we'll understand *how* it's built and *what* it offers — you'll be surprised how little code
it takes.

**Good news:** this beat doesn't need Ollama — MCP is about tools/data, not the language model.

## Step 1 — How a function becomes an MCP tool

We use `FastMCP`, a helper that makes writing a server almost trivial. You create a server
object and decorate ordinary Python functions with `@mcp.tool()`:

```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("orbital")

@mcp.tool()
def get_sensor(signal: str) -> str:
    """Get the latest reading for a colony sensor."""
    return tools.get_telemetry(signal, when="latest")
```

Here's the elegant part: the function's **docstring** becomes the tool's description, and its
**type hints** become the argument schema. So the same information that documents the function
for a human also tells an AI client how to call it. Write clear docstrings and you get a good
MCP tool for free.

👉 **Open `orbital_mcp_server.py` now** and read the five `@mcp.tool()` functions and the one
`@mcp.resource(...)`. Notice they just call the `shared/tools.py` functions from Module 5.

## Step 2 — Inspect what the server exposes

We can load the server object and ask it to list its tools *without* starting the network
transport — handy for a quick look. Each tool reports the name and description a client would
see.

In [ ]:
import importlib.util, os
# Load the server file as a module so we can introspect it.
spec = importlib.util.spec_from_file_location("srv", os.path.abspath("orbital_mcp_server.py"))
srv = importlib.util.module_from_spec(spec); spec.loader.exec_module(srv)

# list_tools() is async, so we 'await' it (Jupyter lets us await at the top level).
tools = await srv.mcp.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

In [ ]:
# TODO: list the resources too (the readable data the server offers).
#   resources = await srv.mcp.list_resources()
#   for r in resources:
#       print(r.uri, "-", r.name)

## Step 3 — Notice the security built into the design

Look again at the tools and how they're written:
- `get_sensor` **rejects unknown signal names** rather than crashing or guessing.
- `raise_alert` **only accepts** green/amber/red.
- `control_valve` is **simulated** and returns a "needs human confirmation" message — it can't
  actually change anything.

A server is a doorway into your systems, so validating inputs and guarding dangerous actions
isn't an afterthought — it's the design. In **6b** we'll connect a real client and call these
over the protocol.

## ✅ Recap

You saw how `@mcp.tool()` turns a documented Python function into a standard, discoverable
tool, and inspected everything the Orbital server offers. Next you'll be the client that
connects to it.

# 6a · Build an MCP server for colony systems

**Core · Live-code · ~25 min**

An MCP **server** exposes tools and resources over a standard protocol. Ours lives in
[`orbital_mcp_server.py`](orbital_mcp_server.py). Let's see how it's built and what it offers.

> Needs the `mcp` package (installed in setup). This beat doesn't need Ollama.

## Turning a function into an MCP tool

With `FastMCP`, a decorator is all it takes:

```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("orbital")

@mcp.tool()
def get_sensor(signal: str) -> str:
    """Get the latest reading for a colony sensor."""
    return tools.get_telemetry(signal, when="latest")
```

The function's **docstring** and **type hints** become the tool's description and schema —
that's what a client (or an LLM) reads to know how to call it.

Open `orbital_mcp_server.py` and read the five `@mcp.tool()` functions and the
`@mcp.resource(...)`.

## Inspect what the server exposes

We can load the server object and ask it for its tools (without starting the transport).

In [ ]:
import importlib.util, os
spec = importlib.util.spec_from_file_location("srv", os.path.abspath("orbital_mcp_server.py"))
srv = importlib.util.module_from_spec(spec); spec.loader.exec_module(srv)

tools = await srv.mcp.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

In [ ]:
# TODO: list the resources too
# resources = await srv.mcp.list_resources()
# for r in resources: print(r.uri, "-", r.name)

## Security by construction

Notice the safe design:
- `get_sensor` rejects unknown signal names.
- `raise_alert` only accepts green/amber/red.
- `control_valve` is **simulated** and returns a "needs human confirmation" message.

In **6b** we'll connect a real client and call these over the protocol.